In [19]:
import hashlib
import os
import platform
import uuid
import sys

# 修改为自己的学号
STUDENT_ID = 20244080131
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"

print(f"[Conda 环境] {CONDA_ENV or '未检测到 Conda 环境'}")
print(f"[学号] {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID] {RUN_ID}")
print(f"[Python] {sys.version.split()[0]}")

[Conda 环境] exp01
[学号] 20244080131
[机器指纹] 73b9d34140867d67
[RUN_ID] 20244080131-73b9d34140867d67
[Python] 3.11.16


# 实验 1：正则表达式的基础匹配

`re.findall(pattern, text)` 会从左到右查找所有非重叠匹配，并返回匹配列表。正则默认区分大小写；`\b` 表示单词边界，`\s` 表示空白字符，`\d` 表示数字。

In [20]:
import re

text = "cat on mat; later: cats and dogs"
patterns = {
    r"\d+": "数字(\\d+)",
    r"\s+": "连续空白(\\s+)",
    r"\bcat\b": "完整单词 cat(\\bcat\\b)",
}

for pattern, description in patterns.items():
    print(f"{description}: {re.findall(pattern, text)!r}")

数字(\d+): []
连续空白(\s+): [' ', ' ', ' ', ' ', ' ', ' ']
完整单词 cat(\bcat\b): ['cat']


本句不含数字，所以 `\d+` 返回空列表。`\s+` 会把每段连续空白作为一个匹配；`\bcat\b` 只匹配独立的 `cat`，不会把 `cats` 中的 `cat` 部分算作完整单词。

In [8]:
text = "cat dog bird CAT"
print("cat|dog:", re.findall(r"cat|dog", text))

sample = "aaaa"
print("a|aa|aaa:", re.findall(r"a|aa|aaa", sample))
print("aaa|aa|a:", re.findall(r"aaa|aa|a", sample))

cat|dog: ['cat', 'dog']
a|aa|aaa: ['a', 'a', 'a', 'a']
aaa|aa|a: ['aaa', 'a']


`cat|dog` 匹配到小写的 `cat` 和 `dog`，不匹配大写的 `CAT`。`findall` 在每个位置按 `|` 从左向右尝试分支：`a|aa|aaa` 先取单个 `a`，所以得到 4 个 `a`；`aaa|aa|a` 优先取较长分支，得到 `aaa` 和剩余的 `a`。

In [9]:
text = "ct cat caat caaat"
quantifiers = [r"ca*t", r"ca+t", r"ca?t", r"ca{2}t", r"ca{1,2}t"]

for pattern in quantifiers:
    print(f"{pattern}: {re.findall(pattern, text)}")

ca*t: ['ct', 'cat', 'caat', 'caaat']
ca+t: ['cat', 'caat', 'caaat']
ca?t: ['ct', 'cat']
ca{2}t: ['caat']
ca{1,2}t: ['cat', 'caat']


量词说明：`*` 表示前一项出现 0 次或多次；`+` 表示 1 次或多次；`?` 表示 0 次或 1 次；`{2}` 表示恰好 2 次；`{1,2}` 表示 1 至 2 次。

# 实验 2：分组、回溯引用与替换

捕获组用括号保存匹配内容，`\1` 可以在同一正则中引用第一个捕获组。`re.sub()` 可用捕获组重新排列或格式化匹配文本。

In [21]:
email_text = "联系:alice@example.com:备选:bob@test.org"
email_pattern = r"([A-Za-z0-9._%+-]+)@([A-Za-z0-9.-]+\.[A-Za-z]{2,})"

for username, domain in re.findall(email_pattern, email_text):
    print(f"用户名: {username}, 域名: {domain}")

用户名: alice, 域名: example.com
用户名: bob, 域名: test.org


第一个捕获组对应 `@` 前的用户名，第二个捕获组对应域名；`findall()` 因为正则中有两个捕获组，所以每个匹配以二元组形式返回。

In [11]:
repeat_text = "cat cat, dog cat, bird bird"
repeat_pattern = r"\b(\w+)\s+\1\b"
repeated_words = [
    (match.group(0), match.group(1))
    for match in re.finditer(repeat_pattern, repeat_text)
]
print("重复单词片段与单词:", repeated_words)

重复单词片段与单词: [('cat cat', 'cat'), ('bird bird', 'bird')]


`\b(\w+)\s+\1\b` 中第一个括号捕获单词，`\1` 要求后一个单词与捕获内容完全相同；因此 `cat cat` 和 `bird bird` 会匹配，`dog cat` 不会。

In [12]:
date_text = "2024-05-01\n2024/05/02\n2024.05.03"
date_pattern = r"\b(\d{4})([-/.])(\d{2})\2(\d{2})\b"
date_replacement = r"\3\2\4\2\1"
converted_dates = re.sub(date_pattern, date_replacement, date_text)

print("转换前:")
print(date_text)
print("转换后:")
print(converted_dates)
assert converted_dates.count("\n") == date_text.count("\n")

转换前:
2024-05-01
2024/05/02
2024.05.03
转换后:
05-01-2024
05/02/2024
05.03.2024


日期正则将年、分隔符、月、日分别捕获，并用第二组的反向引用保证同一日期内分隔符一致；替换为月、原分隔符、日、原分隔符、年。代码还检查转换前后换行符数量一致。

In [18]:
phone_text = "联系我:123-456-7890 或 987.654.3210"
phone_pattern = r"\b(\d{3})[-.](\d{3})[-.](\d{4})\b"
phone_replacement = r"(\1) \2-\3"
print(re.sub(phone_pattern, phone_replacement, phone_text))

联系我:(123) 456-7890 或 (987) 654-3210


# 实验 3：使用 BeautifulSoup 解析 HTML



In [15]:
from bs4 import BeautifulSoup

html_doc = """
<html>
<head><title>实验课程</title></head>
<body>
  <h1>数据分析</h1>
  <p class="course">第六讲：正则表达式</p>
  <p class="course">第七讲：结构化数据</p>
  <a href="https://example.org/lecture06">讲义六</a>
  <a href="https://example.org/lecture07">讲义七</a>
</body>
</html>
"""
soup = BeautifulSoup(html_doc, "html.parser")

page_title = soup.title.get_text(strip=True)
heading = soup.find("h1")
print("页面标题:", page_title)
print("h1 标签名:", heading.name)
print("h1 文本:", heading.get_text(strip=True))

course_paragraphs = soup.find_all("p", class_="course")
print("课程段落:")
for paragraph in course_paragraphs:
    print("-", paragraph.get_text(strip=True))

links = soup.find_all("a")
print("链接:")
for link in links:
    print(f"- 文本: {link.get_text(strip=True)}, href: {link.get('href')}")

print("整页可见文本:", soup.get_text(separator=" | ", strip=True))
print("单个段落文本:", course_paragraphs[0].get_text(strip=True))
print("单个链接文本:", links[0].get_text(strip=True))

页面标题: 实验课程
h1 标签名: h1
h1 文本: 数据分析
课程段落:
- 第六讲：正则表达式
- 第七讲：结构化数据
链接:
- 文本: 讲义六, href: https://example.org/lecture06
- 文本: 讲义七, href: https://example.org/lecture07
整页可见文本: 实验课程 | 数据分析 | 第六讲：正则表达式 | 第七讲：结构化数据 | 讲义六 | 讲义七
单个段落文本: 第六讲：正则表达式
单个链接文本: 讲义六
